Import libraries

In [1]:
import numpy as np
import torch
import random
random.seed(4101)
torch.manual_seed(4101)
import pandas as pd

from data_pipeline import build_pipeline, inverse_transform, load_scaler
from vae_module import VAEConfig, train_vae, encode, decode, load_vae
from timegan_module import train_timegan, generate, load_timegan, TimeGANConfig

Import data

In [2]:
wti_df = pd.read_csv('../data/wti_prices.csv', index_col=0, parse_dates=['Date'])
wti_df = np.log(wti_df).diff().dropna()
cols = list(wti_df.columns)
print(wti_df.shape)
print(cols)

(480, 1)
['WTI_price']


In [3]:
predictors = pd.read_csv('../data/predictor_data.csv', index_col=0, parse_dates=['Date'])
cols = list(predictors.columns)
print(predictors.shape)
print(cols)

(481, 8)
['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD']


In [4]:
# Combine TB3MS and WTI data by month (index)
combined_df = predictors.join(wti_df, how='inner').sort_index()

# Optional: set a clear index name
combined_df.index.name = "month"

#filter data up to 2014-01-01 for training data
combined_df = combined_df[combined_df.index <= '2014-01-01']

print(combined_df.shape)
print(combined_df.isna().sum())

(336, 9)
CPI          0
TB3MS        0
M1           0
M2           0
AUD_USD      0
CAD_USD      0
NZD_USD      0
ZAR_USD      0
WTI_price    0
dtype: int64


Stationarity

In [5]:
from statsmodels.tsa.stattools import adfuller

# For each column we perform adf test
for column in combined_df.columns:
    result = adfuller(combined_df[column].dropna())
    print(f"ADF Statistic for {column}: {result[0]}")
    print(f"p-value for {column}: {result[1]}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value}")

# We see that for all columns except WTI price it is not stationary, so we take log difference to make it stationary, for every column except WTI price, which is already stationary
for column in combined_df.columns:
    if column != 'WTI_price':
        combined_df[column] = np.log(combined_df[column]).diff().dropna()



ADF Statistic for CPI: -0.349125611730296
p-value for CPI: 0.9182527489186103
Critical Values:
   1%: -3.4507587628808922
   5%: -2.870530068560499
   10%: -2.5715597727381647
ADF Statistic for TB3MS: -1.928922065668171
p-value for TB3MS: 0.3185799385364664
Critical Values:
   1%: -3.450632157720528
   5%: -2.870474482366864
   10%: -2.5715301325443787
ADF Statistic for M1: 3.0977607356548615
p-value for M1: 1.0
Critical Values:
   1%: -3.4503836022181056
   5%: -2.8703653471616826
   10%: -2.571471939191249
ADF Statistic for M2: 3.192138306640184
p-value for M2: 1.0
Critical Values:
   1%: -3.4510167751522642
   5%: -2.87064334231426
   10%: -2.5716201744283174
ADF Statistic for AUD_USD: -1.8801382634400816
p-value for AUD_USD: 0.3414616037894379
Critical Values:
   1%: -3.4502011472639724
   5%: -2.8702852297358983
   10%: -2.5714292194077513
ADF Statistic for CAD_USD: -1.298698392514305
p-value for CAD_USD: 0.6297675900976204
Critical Values:
   1%: -3.450081345901191
   5%: -2.8702

In [6]:
cleaned_df = combined_df.dropna()

In [7]:
from data_pipeline import build_pipeline, inverse_transform, load_scaler, PipelineConfig
import random
random.seed(4101)

# Build windowed dataset from raw training DataFrame
windows, scaler = build_pipeline(
    df=cleaned_df,            # rows = timesteps, columns = variables
    scaler_save_path="scaler.pkl"
)
# windows shape: (num_windows, window_length, num_variables)

[Pipeline] Observations: 335 | Variables: 9
[Pipeline] Columns: ['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD', 'WTI_price']
[Pipeline] Window length: 24 | Stride: 1
[Pipeline] Windows to be created: 312
[Pipeline] Scaler: minmax
------------------------------------------------------------
[Pipeline] Normalisation complete.
[Pipeline] Data range after scaling: [0.0000, 1.0000]
[Pipeline] Windows created: 312
[Pipeline] Output shape: (312, 24, 9)  (num_windows, window_length, num_variables)
[Pipeline] Scaler saved to: scaler.pkl
------------------------------------------------------------
[Pipeline] Pipeline complete. Ready for Module 2 (VAE).


In [8]:
config_vae = VAEConfig(
    latent_dim=4,
    hidden_dim=64,
    num_epochs=400,
    kl_warmup_epochs=50,
    kl_weight_max=0.005,
    learning_rate=1e-4,
    verbose_every=25,
    random_seed=4101,
)
model_vae, history = train_vae(
    data=windows,
    config=config_vae,
    save_path="vae_checkpoint.pt"
)
model_vae = load_vae("vae_checkpoint.pt")

# Encode to latent sequences — shape (312, 24, 8)
latent_sequences = encode(windows, model_vae, use_mean=True)
print("Latent sequences shape:", latent_sequences.shape)

[VAE] Device: cuda
[VAE] Input dim: 9 | Latent dim: 4 | Hidden dim: 64
[VAE] Windows: 312 | Window length: 24 | Variables: 9
[VAE] Train windows: 266 | Val windows: 46
[VAE] KL warmup: 50 epochs | Max KL weight: 0.005
------------------------------------------------------------
[VAE] Epoch    1/400 | KL weight: 0.0001 | Train — Total: 0.289295, Recon: 0.289255, KL: 0.400000 | Val   — Total: 0.285911, Recon: 0.285871, KL: 0.400000
[VAE] Epoch   25/400 | KL weight: 0.0025 | Train — Total: 0.070650, Recon: 0.069650, KL: 0.400000 | Val   — Total: 0.077606, Recon: 0.076606, KL: 0.400000
[VAE] Epoch   50/400 | KL weight: 0.0050 | Train — Total: 0.034074, Recon: 0.031580, KL: 0.498692 | Val   — Total: 0.039075, Recon: 0.036514, KL: 0.512349
[VAE] Epoch   75/400 | KL weight: 0.0050 | Train — Total: 0.024807, Recon: 0.021442, KL: 0.673008 | Val   — Total: 0.030205, Recon: 0.026723, KL: 0.696431
[VAE] Epoch  100/400 | KL weight: 0.0050 | Train — Total: 0.020651, Recon: 0.017641, KL: 0.601908 | V

In [9]:
latent = encode(windows, model_vae, use_mean=True)
latent_flat = latent.reshape(-1, latent.shape[-1])
print("Latent mean:", latent_flat.mean(axis=0).round(4))
print("Latent std: ", latent_flat.std(axis=0).round(4))

import torch
tensor = torch.tensor(windows, dtype=torch.float32)
flat = tensor.reshape(-1, tensor.shape[-1])
with torch.no_grad():
    mu, log_var = model_vae.encoder(flat)
print("posterior std min:", torch.exp(0.5 * log_var).min().item())
print("posterior std max:", torch.exp(0.5 * log_var).max().item())

Latent mean: [-0.0172  0.1397 -0.1186 -0.0395]
Latent std:  [0.2755 0.1736 0.1482 0.2581]
posterior std min: 0.38342443108558655
posterior std max: 0.936032772064209


Using my own coded timegan for latent

In [10]:
import importlib
import timegan_module
importlib.reload(timegan_module)
from timegan_module import train_timegan, generate, load_timegan, TimeGANConfig
import torch

config_gan_hybrid = TimeGANConfig(
    hidden_dim=32,
    num_layers=2,
    noise_dim=4,
    embedding_dim=32,
    num_epochs=1000,
    batch_size=32,
    learning_rate=1e-4,
    gamma=1.0,
    phase_split=(0.2, 0.2, 0.6),
    early_stopping_patience=10,
    eval_every_n_epochs=50,
    verbose_every=50,
    random_seed=4101,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

model_gan_hybrid, history_hybrid = train_timegan(
    data=latent_sequences,
    config=config_gan_hybrid,
    save_path="timegan_hybrid_checkpoint.pt",
    plot_save_path="timegan_hybrid_loss.png",
)

[TimeGAN] Device: cuda
[TimeGAN] Input dim: 4 | Embedding dim: 32 | Hidden dim: 32 | Noise dim: 4
[TimeGAN] Windows: 312 | Seq length: 24
[TimeGAN] Total epochs: 1000 | Phase split — Embed: 200 | Supervised: 200 | Joint: 600
[TimeGAN] Early stopping: enabled (patience=10)

[Phase 1/3] Embedding Pretraining (200 epochs)
----------------------------------------------------------------------
  [Embedding] Epoch    1/200 | Loss: 1.090878
  [Embedding] Epoch   50/200 | Loss: 0.248000
  [Embedding] Epoch  100/200 | Loss: 0.034326
  [Embedding] Epoch  150/200 | Loss: 0.025615
  [Embedding] Epoch  200/200 | Loss: 0.023227

[Phase 2/3] Supervised Pretraining (200 epochs)
----------------------------------------------------------------------
  [Supervised] Epoch    1/200 | Loss: 0.009674
  [Supervised] Epoch   50/200 | Loss: 0.007144
  [Supervised] Epoch  100/200 | Loss: 0.005570
  [Supervised] Epoch  150/200 | Loss: 0.004934
  [Supervised] Epoch  200/200 | Loss: 0.004688

[Phase 3/3] Joint GAN 

In [11]:
from timegan_module import generate, load_timegan
from vae_module import decode, load_vae
from data_pipeline import inverse_transform, load_scaler
from evaluation_module import EvaluationConfig, evaluate_all

model_gan = load_timegan("timegan_hybrid_checkpoint.pt")
model_vae = load_vae("vae_checkpoint.pt")
scaler = load_scaler("scaler.pkl")

synthetic_latent     = generate(312, model_gan)
synthetic_normalised = decode(synthetic_latent, model_vae)

config_eval = EvaluationConfig(
    n_lags=12,
    random_seed=4101,
    report_save_path="results/results_hybrid_v2.txt",
    json_save_path="results/results_hybrid_v2.json",
    figure_save_path="results/figure_hybrid_v2.png",
)

results = evaluate_all(
    real=windows,
    synthetic=synthetic_normalised,
    config=config_eval,
    variable_names=["CPI","TB3MS","M1","M2",
                    "AUD_USD","CAD_USD","NZD_USD","ZAR_USD","WTI_price"],
)


  EVALUATION SUITE — running all metrics
  Real: 312 windows | Synthetic: 312 windows
  Variables: ['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD', 'WTI_price']

[1/6] Computing statistical moments...
  CPI: real_std=0.0831 | synth_std=0.0006 | std_ratio=0.0071
  TB3MS: real_std=0.0691 | synth_std=0.0001 | std_ratio=0.0019
  M1: real_std=0.0948 | synth_std=0.0003 | std_ratio=0.0034
  M2: real_std=0.1328 | synth_std=0.0001 | std_ratio=0.0010
  AUD_USD: real_std=0.1089 | synth_std=0.0004 | std_ratio=0.0040
  CAD_USD: real_std=0.0921 | synth_std=0.0002 | std_ratio=0.0020
  NZD_USD: real_std=0.1498 | synth_std=0.0009 | std_ratio=0.0060
  ZAR_USD: real_std=0.1073 | synth_std=0.0004 | std_ratio=0.0039
  WTI_price: real_std=0.1137 | synth_std=0.0007 | std_ratio=0.0062

[2/6] Computing ACF comparison...
  Mean ACF MAE: 0.2027
  CPI: ACF MAE = 0.2298
  TB3MS: ACF MAE = 0.1180
  M1: ACF MAE = 0.2749
  M2: ACF MAE = 0.2461
  AUD_USD: ACF MAE = 0.1397
  CAD_USD: ACF MAE = 

using my own on normal timeGAN

In [12]:
importlib.reload(timegan_module)
from timegan_module import train_timegan, generate, load_timegan, TimeGANConfig

config_gan_standard = TimeGANConfig(
    hidden_dim=32,
    num_layers=2,
    noise_dim=9,        # match raw variable count
    embedding_dim=32,
    num_epochs=1000,
    batch_size=32,
    learning_rate=1e-4,
    gamma=1.0,
    phase_split=(0.2, 0.2, 0.6),
    early_stopping_patience=10,
    eval_every_n_epochs=50,
    verbose_every=50,
    random_seed=4101,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

model_gan_standard, history_standard = train_timegan(
    data=windows,       # raw normalised windows, no VAE
    config=config_gan_standard,
    save_path="timegan_standard_checkpoint.pt",
    plot_save_path="timegan_standard_loss.png",
)

[TimeGAN] Device: cuda
[TimeGAN] Input dim: 9 | Embedding dim: 32 | Hidden dim: 32 | Noise dim: 9
[TimeGAN] Windows: 312 | Seq length: 24
[TimeGAN] Total epochs: 1000 | Phase split — Embed: 200 | Supervised: 200 | Joint: 600
[TimeGAN] Early stopping: enabled (patience=10)

[Phase 1/3] Embedding Pretraining (200 epochs)
----------------------------------------------------------------------
  [Embedding] Epoch    1/200 | Loss: 2.844575
  [Embedding] Epoch   50/200 | Loss: 0.166904
  [Embedding] Epoch  100/200 | Loss: 0.109795
  [Embedding] Epoch  150/200 | Loss: 0.081157
  [Embedding] Epoch  200/200 | Loss: 0.073647

[Phase 2/3] Supervised Pretraining (200 epochs)
----------------------------------------------------------------------
  [Supervised] Epoch    1/200 | Loss: 0.005977
  [Supervised] Epoch   50/200 | Loss: 0.005076
  [Supervised] Epoch  100/200 | Loss: 0.004875
  [Supervised] Epoch  150/200 | Loss: 0.004487
  [Supervised] Epoch  200/200 | Loss: 0.004558

[Phase 3/3] Joint GAN 

KeyboardInterrupt: 

In [ ]:
# 2. Standard TimeGAN
model_gan_standard = load_timegan("timegan_standard_checkpoint.pt")
synthetic_timegan  = generate(312, model_gan_standard)
# already in normalised space — no VAE decode needed

# 3. VAE-TimeGAN hybrid
model_gan_hybrid  = load_timegan("timegan_hybrid_checkpoint.pt")
synthetic_latent  = generate(312, model_gan_hybrid)
synthetic_hybrid  = decode(synthetic_latent, model_vae)

print("TimeGAN shape:  ", synthetic_timegan.shape)
print("Hybrid shape:   ", synthetic_hybrid.shape)

Raw output shape: (312, 24, 8)
Synthetic latent shape: (312, 24, 8)
Synthetic normalised shape: (312, 24, 9)
Synthetic original shape: (312, 24, 9)


In [ ]:
from evaluation_module import EvaluationConfig, evaluate_all

variable_names = ["CPI","TB3MS","M1","M2",
                  "AUD_USD","CAD_USD","NZD_USD","ZAR_USD","WTI_price"]

# Jittering
config_eval = EvaluationConfig(
    n_lags=12,
    random_seed=4101,
    report_save_path="results_timegan.txt",
    json_save_path="results_timegan.json",
    figure_save_path="figure_timegan.png",
)
results_timegan = evaluate_all(
    real=windows,
    synthetic=synthetic_timegan,
    config=config_eval,
    variable_names=variable_names,
)

# VAE-TimeGAN hybrid
config_eval.report_save_path = "results_hybrid.txt"
config_eval.json_save_path   = "results_hybrid.json"
config_eval.figure_save_path = "figure_hybrid.png"
results_hybrid = evaluate_all(
    real=windows,
    synthetic=synthetic_hybrid,
    config=config_eval,
    variable_names=variable_names,
)


  EVALUATION SUITE — running all metrics
  Real: 312 windows | Synthetic: 312 windows
  Variables: ['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD', 'WTI_price']

[1/6] Computing statistical moments...
  CPI: real_std=0.0831 | synth_std=0.0001 | std_ratio=0.0009
  TB3MS: real_std=0.0691 | synth_std=0.0134 | std_ratio=0.1940
  M1: real_std=0.0948 | synth_std=0.0005 | std_ratio=0.0050
  M2: real_std=0.1328 | synth_std=0.0002 | std_ratio=0.0012
  AUD_USD: real_std=0.1089 | synth_std=0.0007 | std_ratio=0.0062
  CAD_USD: real_std=0.0921 | synth_std=0.0015 | std_ratio=0.0160
  NZD_USD: real_std=0.1498 | synth_std=0.0027 | std_ratio=0.0179
  ZAR_USD: real_std=0.1073 | synth_std=0.0057 | std_ratio=0.0529
  WTI_price: real_std=0.1137 | synth_std=0.0028 | std_ratio=0.0250

[2/6] Computing ACF comparison...
  Mean ACF MAE: 0.0986
  CPI: ACF MAE = 0.0271
  TB3MS: ACF MAE = 0.1153
  M1: ACF MAE = 0.1752
  M2: ACF MAE = 0.1286
  AUD_USD: ACF MAE = 0.0552
  CAD_USD: ACF MAE = 